In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

In [2]:
def calcola_similarita_jaccard(testo1, testo2):
    set1 = set(str(testo1).lower().split())
    set2 = set(str(testo2).lower().split())
    if not set1 or not set2:
        return 0.0
    return len(set1.intersection(set2)) / len(set1.union(set2))

def build_dpo_parquet(
    filepath, 
    output_parquet_path, 
    min_delta=0.3, 
    similarity_threshold=0.85,
    max_pairs_per_prompt=None,
    conversational_format=True  # True per Opzione A, False per Opzione B
):
    df = pd.read_csv(filepath)
    df = df.dropna(subset=['question', 'answer', 'average_rank_os'])
    
    # Analisi preliminare veloce
    risposte_per_prompt = df.groupby('question').size()
    print(f"Prompt unici rilevati: {len(risposte_per_prompt)}")
    print(f"Media risposte per prompt: {risposte_per_prompt.mean():.2f}\n")
    
    dpo_list = []
    grouped = df.groupby('question')
    
    for question, group in grouped:
        records = group[['answer', 'average_rank_os', 'model_name']].to_dict('records')
        prompt_pairs = []
        
        for r1, r2 in combinations(records, 2):
            score1 = r1['average_rank_os']
            score2 = r2['average_rank_os']
            delta = abs(score1 - score2)
            
            if delta < min_delta:
                continue
                
            if score1 > score2:
                chosen_rec, rejected_rec = r1, r2
            else:
                chosen_rec, rejected_rec = r2, r1
                
            sim = calcola_similarita_jaccard(chosen_rec['answer'], rejected_rec['answer'])
            if sim > similarity_threshold:
                continue
            
            # Formattazione in base alla scelta del formato
            if conversational_format:
                # Opzione A: Struttura a messaggi (consigliata per TRL)
                prompt_data = [{"role": "user", "content": question}]
                chosen_data = [{"role": "assistant", "content": chosen_rec['answer']}]
                rejected_data = [{"role": "assistant", "content": rejected_rec['answer']}]
            else:
                # Opzione B: Testo piatto con ChatML esplicito per Qwen
                prompt_data = f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
                chosen_data = f"{chosen_rec['answer']}<|im_end|>"
                rejected_data = f"{rejected_rec['answer']}<|im_end|>"
                
            prompt_pairs.append({
                "prompt": prompt_data,
                "chosen": chosen_data,
                "rejected": rejected_data,
                "delta": float(delta)
            })
        
        if max_pairs_per_prompt is not None:
            prompt_pairs = sorted(prompt_pairs, key=lambda x: x['delta'], reverse=True)
            prompt_pairs = prompt_pairs[:max_pairs_per_prompt]
            
        dpo_list.extend(prompt_pairs)
        
    # Conversione in DataFrame Pandas
    dpo_df = pd.DataFrame(dpo_list)
    
    # Rimuoviamo la colonna di comodo 'delta' prima del salvataggio se non ti serve nel dataset finale
    if not dpo_df.empty:
        dpo_df = dpo_df.drop(columns=['delta'])
    
    print(f"Triple DPO totali generate: {len(dpo_df)}")
    
    # Salvataggio in Parquet
    dpo_df.to_parquet(output_parquet_path, index=False, engine='pyarrow')
    print(f"Dataset salvato con successo in: {output_parquet_path}")
    
    return dpo_df

In [3]:
# --- 2. CONFIGURAZIONE DEI PERCORSI E ESECUZIONE ---

# Sostituisci questo percorso con quello reale del tuo file CSV su Kaggle
percorso_csv_ingresso = '/kaggle/input/datasets/lorenzosalis/autobench-run5-os/Run5_data_OS.csv' 

# Salviamo l'output nella cartella di lavoro di Kaggle
percorso_parquet_uscita = '/kaggle/working/dpo_dataset_qwen.parquet'

# Avvio dell'elaborazione
df_dpo_risultato = build_dpo_parquet(
    filepath=percorso_csv_ingresso,
    output_parquet_path=percorso_parquet_uscita,
    min_delta=0.3,                  # Soglia di preferenza minima
    similarity_threshold=0.85,      # Filtro somiglianza testuale
    conversational_format=True      # True per formato a messaggi [user/assistant]
 )

Prompt unici rilevati: 503
Media risposte per prompt: 10.70

Triple DPO totali generate: 21868
Dataset salvato con successo in: /kaggle/working/dpo_dataset_qwen.parquet
